# Experiment List:
1. Baseline Performance of Neural Models
2. Wisdom of The Crowd Experiment
3. Using "Polarization" as a Feature for Toxicity Detection
4. Incorporating Demographic Information
5. Combining Polarization and Demographic Information for Toxicity Detection

## 1. Baseline Performance of Neural Models

### For IndoBERTweet and NusaBERT

In [ ]:
import pandas as pd
import ast
import os
import numpy as np
from sklearn.model_selection import StratifiedKFold
from transformers import Trainer, TrainingArguments, BertForSequenceClassification, BertTokenizer
import torch

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, average_precision_score

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    # Accuracy
    accuracy = accuracy_score(labels, preds)

    # Macro F1, Precision, and Recall
    macro_f1 = f1_score(labels, preds, average='macro')
    precision = precision_score(labels, preds, average='macro')
    recall = recall_score(labels, preds, average='macro')

    # Class-1 only metrics (positive class)
    precision_class_1 = precision_score(labels, preds, pos_label=1)
    recall_class_1 = recall_score(labels, preds, pos_label=1)
    f1_class_1 = f1_score(labels, preds, pos_label=1)

    # Class-0 only metrics (negative class)
    precision_class_0 = precision_score(labels, preds, pos_label=0)
    recall_class_0 = recall_score(labels, preds, pos_label=0)
    f1_class_0 = f1_score(labels, preds, pos_label=0)

    # ROC-AUC score for binary classification
    try:
        # Compute the ROC AUC score for binary classification directly
        roc_auc = roc_auc_score(labels, preds)
    except ValueError:
        # In case there's an issue with the labels or predictions (e.g., all labels are the same)
        roc_auc = 0.5  # This would represent random classification if AUC can't be computed

    # Precision-Recall AUC
    precision_recall_auc = average_precision_score(labels, preds)

    return {
        'accuracy': accuracy,
        'macro_f1': macro_f1,
        'precision': precision,
        'recall': recall,
        'precision_class_1': precision_class_1,
        'recall_class_1': recall_class_1,
        'f1_class_1': f1_class_1,
        'precision_class_0': precision_class_0,
        'recall_class_0': recall_class_0,
        'f1_class_0': f1_class_0,
        'roc_auc': roc_auc,
        'precision_recall_auc': precision_recall_auc,
    }

def wisdom_text_handler(merged_df):
    texts = merged_df['text'].tolist()
    labels = merged_df['label'].tolist()
    annot_counts = merged_df['annotator_count'].astype(int).tolist()
    return texts, labels, annot_counts

def wisdom_any_text_handler(merged_df):
    texts = merged_df['text'].tolist()
    merged_df['polarized'] = merged_df['polarized'].apply(lambda x: [int(y) for y in ast.literal_eval(x)])
    merged_df['polarized_value'] = merged_df['polarized'].apply(lambda x: sum(x)/len(x))
    merged_df['any_label'] = merged_df['polarized_value'].apply(lambda x: 1 if x > 0 else 0) 
    any_label = merged_df['any_label'].tolist()
    labels = merged_df['label'].tolist()
    annot_counts = merged_df['annotator_count'].astype(int).tolist()
    return texts, labels, annot_counts, any_label

def train_wisdom_bert_pipeline(model_path: str, merged_df, output_dir: str, approach: str = "single"):

    # Handle texts and labels:
    texts, labels, annot_counts = wisdom_text_handler(merged_df)

    # Create output directory
    os.makedirs(output_dir, exist_ok=True)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    metrics_list = []

    for fold, (train_index, test_index) in enumerate(skf.split(texts, labels)):
        train_annot_counts = np.array(annot_counts)[train_index]
        train_texts, test_texts = np.array(texts)[train_index], np.array(texts)[test_index]
        train_labels, test_labels = np.array(labels)[train_index], np.array(labels)[test_index]
        target_annot_count = 1
        if approach == "single":
          train_indices = np.where(train_annot_counts == target_annot_count)[0]
          train_texts = np.array(train_texts)[train_indices]
          train_labels = np.array(train_labels)[train_indices]
        elif approach == "more":
          train_indices = np.where(train_annot_counts != target_annot_count)[0]
          train_texts = np.array(train_texts)[train_indices]
          train_labels = np.array(train_labels)[train_indices]
        else:
          raise ValueError(f"Invalid approach: {approach}")
        # Tokenize
        tokenizer = BertTokenizer.from_pretrained(model_path)
        train_encodings = tokenizer(train_texts.tolist(), truncation=True, padding=True, max_length=512, return_tensors='pt')
        test_encodings = tokenizer(test_texts.tolist(), truncation=True, padding=True, max_length=512, return_tensors='pt')

        train_dataset = Dataset(train_encodings, train_labels)
        test_dataset = Dataset(test_encodings, test_labels)

        model = BertForSequenceClassification.from_pretrained(model_path, num_labels=len(set(labels)))

        training_args = TrainingArguments(
            output_dir=os.path.join(output_dir, f"model_fold_{fold}"),
            evaluation_strategy="epoch",
            per_device_train_batch_size=16,
            per_device_eval_batch_size=64,
            num_train_epochs=3,
            logging_dir=os.path.join(output_dir, f"logs_fold_{fold}"),
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=test_dataset,
            compute_metrics=compute_metrics,
        )

        # Train and evaluate
        trainer.train()
        metrics = trainer.evaluate()
        metrics_list.append(metrics)

        # Save model
        model.save_pretrained(os.path.join(output_dir, f"model_fold_{fold}"))
        tokenizer.save_pretrained(os.path.join(output_dir, f"model_fold_{fold}"))

        # Save performance report
        pd.DataFrame([metrics]).to_csv(os.path.join(output_dir, f"performance_fold_{fold}.csv"), index=False)

    # Calculate average performance metrics
    avg_metrics = {metric: np.mean([m[metric] for m in metrics_list]) for metric in metrics_list[0]}
    pd.DataFrame([avg_metrics]).to_csv(os.path.join(output_dir, "average_performance.csv"), index=False)

# Dataset class to handle encoding
class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = [int(label) for label in labels] 

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

### For Multi-e5

In [ ]:
import pandas as pd
import ast
import os
import numpy as np
import torch
from sklearn.model_selection import StratifiedKFold
from transformers import Trainer, TrainingArguments, XLMRobertaForSequenceClassification, XLMRobertaTokenizer

def baseline_text_handler(merged_df):
    texts = merged_df['text'].tolist()
    labels = merged_df['label'].tolist()
    return texts, labels

def train_baseline_XLMRoberta_pipeline(model_path: str, merged_df, output_dir: str):

    # Handle texts and labels:
    texts, labels = baseline_text_handler(merged_df)

    # Create output directory
    os.makedirs(output_dir, exist_ok=True)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    metrics_list = []

    for fold, (train_index, test_index) in enumerate(skf.split(texts, labels)):
        train_texts, test_texts = np.array(texts)[train_index], np.array(texts)[test_index]
        train_labels, test_labels = np.array(labels)[train_index], np.array(labels)[test_index]

        # Tokenize
        tokenizer = XLMRobertaTokenizer.from_pretrained(model_path)
        train_encodings = tokenizer(train_texts.tolist(), truncation=True, padding=True, max_length=512, return_tensors='pt')
        test_encodings = tokenizer(test_texts.tolist(), truncation=True, padding=True, max_length=512, return_tensors='pt')

        train_dataset = Dataset(train_encodings, train_labels)
        test_dataset = Dataset(test_encodings, test_labels)

        model = XLMRobertaForSequenceClassification.from_pretrained(model_path, num_labels=len(set(labels)))

        training_args = TrainingArguments(
            output_dir=os.path.join(output_dir, f"model_fold_{fold}"),
            evaluation_strategy="epoch",
            per_device_train_batch_size=16,
            per_device_eval_batch_size=64,
            num_train_epochs=3,
            logging_dir=os.path.join(output_dir, f"logs_fold_{fold}"),
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=test_dataset,
            compute_metrics=compute_metrics,
        )

        # Train and evaluate
        trainer.train()
        metrics = trainer.evaluate()
        metrics_list.append(metrics)

        # Save model
        model.save_pretrained(os.path.join(output_dir, f"model_fold_{fold}"))
        tokenizer.save_pretrained(os.path.join(output_dir, f"model_fold_{fold}"))

        # Save performance report
        pd.DataFrame([metrics]).to_csv(os.path.join(output_dir, f"performance_fold_{fold}.csv"), index=False)

    # Final training on the whole dataset
    tokenizer = XLMRobertaTokenizer.from_pretrained(model_path)
    encodings = tokenizer(texts, truncation=True, padding=True, max_length=512, return_tensors='pt')
    dataset = Dataset(encodings, labels)

    model = XLMRobertaForSequenceClassification.from_pretrained(model_path, num_labels=len(set(labels)))

    training_args = TrainingArguments(
        output_dir=os.path.join(output_dir, "final_model"),
        num_train_epochs=3,
        per_device_train_batch_size=16,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset,
    )

    trainer.train()
    model.save_pretrained(os.path.join(output_dir, "final_model"))
    tokenizer.save_pretrained(os.path.join(output_dir, "final_model"))

    # Calculate average performance metrics
    avg_metrics = {metric: np.mean([m[metric] for m in metrics_list]) for metric in metrics_list[0]}
    pd.DataFrame([avg_metrics]).to_csv(os.path.join(output_dir, "average_performance.csv"), index=False)

## 2. Wisdom of The Crowd Experiment
This experiment consists of two main functions: `train_wisdom_bert_pipeline` and `train_wisdom_bert_normalized_sampling_pipeline`.
1. train_wisdom_bert_pipeline does not maintain label distribution across experiments (i.e., the Single subset has a different ratio of toxic to non-toxic texts compared to the More subset).
2. train_wisdom_bert_normalized_sampling_pipeline applies upsampling to ensure that the Single subset has a distribution of toxic to non-toxic texts that is nearly identical to the More subset.

In [ ]:
import pandas as pd
import ast
import os
import numpy as np
from sklearn.model_selection import StratifiedKFold
from transformers import Trainer, TrainingArguments, BertForSequenceClassification, BertTokenizer
import torch

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, average_precision_score

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    # Accuracy
    accuracy = accuracy_score(labels, preds)

    # Macro F1, Precision, and Recall
    macro_f1 = f1_score(labels, preds, average='macro')
    precision = precision_score(labels, preds, average='macro')
    recall = recall_score(labels, preds, average='macro')

    # Class-1 only metrics (positive class)
    precision_class_1 = precision_score(labels, preds, pos_label=1)
    recall_class_1 = recall_score(labels, preds, pos_label=1)
    f1_class_1 = f1_score(labels, preds, pos_label=1)

    # Class-0 only metrics (negative class)
    precision_class_0 = precision_score(labels, preds, pos_label=0)
    recall_class_0 = recall_score(labels, preds, pos_label=0)
    f1_class_0 = f1_score(labels, preds, pos_label=0)

    # ROC-AUC score for binary classification
    try:
        # Compute the ROC AUC score for binary classification directly
        roc_auc = roc_auc_score(labels, preds)
    except ValueError:
        # In case there's an issue with the labels or predictions (e.g., all labels are the same)
        roc_auc = 0.5  # This would represent random classification if AUC can't be computed

    # Precision-Recall AUC
    precision_recall_auc = average_precision_score(labels, preds)

    return {
        'accuracy': accuracy,
        'macro_f1': macro_f1,
        'precision': precision,
        'recall': recall,
        'precision_class_1': precision_class_1,
        'recall_class_1': recall_class_1,
        'f1_class_1': f1_class_1,
        'precision_class_0': precision_class_0,
        'recall_class_0': recall_class_0,
        'f1_class_0': f1_class_0,
        'roc_auc': roc_auc,
        'precision_recall_auc': precision_recall_auc,
    }

def wisdom_text_handler(merged_df):
    texts = merged_df['text'].tolist()
    labels = merged_df['label'].tolist()
    annot_counts = merged_df['annotator_count'].astype(int).tolist()
    return texts, labels, annot_counts

def wisdom_any_text_handler(merged_df):
    texts = merged_df['text'].tolist()
    merged_df['polarized'] = merged_df['polarized'].apply(lambda x: [int(y) for y in ast.literal_eval(x)])
    merged_df['polarized_value'] = merged_df['polarized'].apply(lambda x: sum(x)/len(x))
    merged_df['any_label'] = merged_df['polarized_value'].apply(lambda x: 1 if x > 0 else 0) 
    any_label = merged_df['any_label'].tolist()
    labels = merged_df['label'].tolist()
    annot_counts = merged_df['annotator_count'].astype(int).tolist()
    return texts, labels, annot_counts, any_label

def train_wisdom_bert_pipeline(model_path: str, merged_df, output_dir: str, approach: str = "single"):

    # Handle texts and labels:
    texts, labels, annot_counts = wisdom_text_handler(merged_df)

    # Create output directory
    os.makedirs(output_dir, exist_ok=True)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    metrics_list = []

    for fold, (train_index, test_index) in enumerate(skf.split(texts, labels)):
        train_annot_counts = np.array(annot_counts)[train_index]
        train_texts, test_texts = np.array(texts)[train_index], np.array(texts)[test_index]
        train_labels, test_labels = np.array(labels)[train_index], np.array(labels)[test_index]
        target_annot_count = 1
        if approach == "single":
          train_indices = np.where(train_annot_counts == target_annot_count)[0]
          train_texts = np.array(train_texts)[train_indices]
          train_labels = np.array(train_labels)[train_indices]
        elif approach == "more":
          train_indices = np.where(train_annot_counts != target_annot_count)[0]
          train_texts = np.array(train_texts)[train_indices]
          train_labels = np.array(train_labels)[train_indices]
        else:
          raise ValueError(f"Invalid approach: {approach}")
        # Tokenize
        tokenizer = BertTokenizer.from_pretrained(model_path)
        train_encodings = tokenizer(train_texts.tolist(), truncation=True, padding=True, max_length=512, return_tensors='pt')
        test_encodings = tokenizer(test_texts.tolist(), truncation=True, padding=True, max_length=512, return_tensors='pt')

        train_dataset = Dataset(train_encodings, train_labels)
        test_dataset = Dataset(test_encodings, test_labels)

        model = BertForSequenceClassification.from_pretrained(model_path, num_labels=len(set(labels)))

        training_args = TrainingArguments(
            output_dir=os.path.join(output_dir, f"model_fold_{fold}"),
            evaluation_strategy="epoch",
            per_device_train_batch_size=16,
            per_device_eval_batch_size=64,
            num_train_epochs=3,
            logging_dir=os.path.join(output_dir, f"logs_fold_{fold}"),
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=test_dataset,
            compute_metrics=compute_metrics,
        )

        # Train and evaluate
        trainer.train()
        metrics = trainer.evaluate()
        metrics_list.append(metrics)

        # Save model
        model.save_pretrained(os.path.join(output_dir, f"model_fold_{fold}"))
        tokenizer.save_pretrained(os.path.join(output_dir, f"model_fold_{fold}"))

        # Save performance report
        pd.DataFrame([metrics]).to_csv(os.path.join(output_dir, f"performance_fold_{fold}.csv"), index=False)

    # Calculate average performance metrics
    avg_metrics = {metric: np.mean([m[metric] for m in metrics_list]) for metric in metrics_list[0]}
    pd.DataFrame([avg_metrics]).to_csv(os.path.join(output_dir, "average_performance.csv"), index=False)

from sklearn.utils import resample

def train_wisdom_bert_normalized_sampling_pipeline(model_path: str, merged_df, output_dir: str, approach: str = "single"):

    # Handle texts and labels:
    texts, labels, annot_counts = wisdom_text_handler(merged_df)
    merged_df["label"] = labels
    merged_df["text"] = texts
    merged_df["annotator_count"] = annot_counts

    # Create output directory
    os.makedirs(output_dir, exist_ok=True)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    metrics_list = []

    for fold, (train_index, test_index) in enumerate(skf.split(texts, labels)):
        train_df = merged_df.iloc[train_index]
        test_df = merged_df.iloc[test_index]

        # Normalize class ratio in train_df
        normalized_dfs = []
        
        if approach == "single":
            subset = train_df[train_df["annotator_count"] == 1]
        elif approach == "more":
            subset = train_df[train_df["annotator_count"] != 1]
    
        # Split into classes
        class_0 = subset[subset["label"] == 0]
        class_1 = subset[subset["label"] == 1]

        # Desired number of class 1 samples to maintain a 1:3 ratio
        target_class_1_count = len(class_0) // 3

        # Resample class 1 (upsample or downsample as needed)
        if len(class_1) > target_class_1_count:
            class_1_resampled = resample(class_1, replace=False, n_samples=target_class_1_count, random_state=42)
        else:
            class_1_resampled = resample(class_1, replace=True, n_samples=target_class_1_count, random_state=42)

        # Combine resampled class 1 with class 0
        normalized_subset = pd.concat([class_0, class_1_resampled])
        normalized_dfs.append(normalized_subset)
        normalized_train_df = pd.concat(normalized_dfs)
        print(normalized_train_df['label'].value_counts())

        # Prepare texts and labels for training
        train_texts = normalized_train_df["text"].tolist()
        train_labels = normalized_train_df["label"].tolist()

        test_texts = test_df["text"].tolist()
        test_labels = test_df["label"].tolist()

        # Tokenize
        tokenizer = BertTokenizer.from_pretrained(model_path)
        train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=512, return_tensors="pt")
        test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=512, return_tensors="pt")

        train_dataset = Dataset(train_encodings, train_labels)
        test_dataset = Dataset(test_encodings, test_labels)

        model = BertForSequenceClassification.from_pretrained(model_path, num_labels=2)

        training_args = TrainingArguments(
            output_dir=os.path.join(output_dir, f"model_fold_{fold}"),
            evaluation_strategy="epoch",
            per_device_train_batch_size=16,
            per_device_eval_batch_size=64,
            num_train_epochs=3,
            logging_dir=os.path.join(output_dir, f"logs_fold_{fold}"),
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=test_dataset,
            compute_metrics=compute_metrics,
        )

        # Train and evaluate
        trainer.train()
        metrics = trainer.evaluate()
        metrics_list.append(metrics)

        # Save model
        model.save_pretrained(os.path.join(output_dir, f"model_fold_{fold}"))
        tokenizer.save_pretrained(os.path.join(output_dir, f"model_fold_{fold}"))

        # Save performance report
        pd.DataFrame([metrics]).to_csv(os.path.join(output_dir, f"performance_fold_{fold}.csv"), index=False)

    # Calculate average performance metrics
    avg_metrics = {metric: np.mean([m[metric] for m in metrics_list]) for metric in metrics_list[0]}
    pd.DataFrame([avg_metrics]).to_csv(os.path.join(output_dir, "average_performance.csv"), index=False)

# Dataset class to handle encoding
class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = [int(label) for label in labels] 

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

## Using "Polarization" as a Feature for Toxicity Detection
Main Function:
`train_featural_bert_pipeline_with_polarized_feature`

Note: 
1. For method = `any`, `bin` and `bin-ceil` method are for pre-eliminary experiments. Both of these methods lead to a worse performing model than `agg`.
2. We attempted to use the English translation of the prompt to incorporate the "Polarization" feature. However, using the English translation leads to a worsening performance, as IndoBERTweet is mainly trained on the Indonesian language.

In [ ]:
import pandas as pd
import ast
import os
import numpy as np
from sklearn.model_selection import StratifiedKFold
from transformers import Trainer, TrainingArguments, BertForSequenceClassification, BertTokenizer
import torch

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, average_precision_score

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    # Accuracy
    accuracy = accuracy_score(labels, preds)

    # Macro F1, Precision, and Recall
    macro_f1 = f1_score(labels, preds, average='macro')
    precision = precision_score(labels, preds, average='macro')
    recall = recall_score(labels, preds, average='macro')

    # Class-1 only metrics (positive class)
    precision_class_1 = precision_score(labels, preds, pos_label=1)
    recall_class_1 = recall_score(labels, preds, pos_label=1)
    f1_class_1 = f1_score(labels, preds, pos_label=1)

    # Class-0 only metrics (negative class)
    precision_class_0 = precision_score(labels, preds, pos_label=0)
    recall_class_0 = recall_score(labels, preds, pos_label=0)
    f1_class_0 = f1_score(labels, preds, pos_label=0)

    # ROC-AUC score for binary classification
    try:
        # Compute the ROC AUC score for binary classification directly
        roc_auc = roc_auc_score(labels, preds)
    except ValueError:
        # In case there's an issue with the labels or predictions (e.g., all labels are the same)
        roc_auc = 0.5  # This would represent random classification if AUC can't be computed

    # Precision-Recall AUC
    precision_recall_auc = average_precision_score(labels, preds)

    return {
        'accuracy': accuracy,
        'macro_f1': macro_f1,
        'precision': precision,
        'recall': recall,
        'precision_class_1': precision_class_1,
        'recall_class_1': recall_class_1,
        'f1_class_1': f1_class_1,
        'precision_class_0': precision_class_0,
        'recall_class_0': recall_class_0,
        'f1_class_0': f1_class_0,
        'roc_auc': roc_auc,
        'precision_recall_auc': precision_recall_auc,
    }

def featural_text_handler(merged_df, method: str = "agg", language: str = "id"):
    """
    Handles toxic and non-toxic text datasets for toxicity classification, 
    applying polarization processing and text formatting.

    Method options:
        1. agg = Aggregate value with a range of [0, 1].
        2. bin = Binarized, values of either 0 or 1 (values of 0.5 converted to 0).
        3. bin-ceil = Binarized, but values of 0.5 converted to 1.
        4. any = Binarized, any value above 0 is converted to 1.

    Language options:
        1. id = Indonesian.
        2. en = English.
    """
    def process_polarized_values(row, method):
        """Processes the polarization values according to the selected method."""
        values = ast.literal_eval(row['polarized']) if isinstance(row['polarized'], str) else row['polarized']
        values = [int(x) for x in values]
        if not values:
            return 0  # Default for missing or empty polarization
        
        agg_value = sum(values) / len(values)
        if method == "agg":
            return agg_value
        elif method == "bin":
            return 1 if agg_value > 0.5 else 0
        elif method == "bin-ceil":
            return 1 if agg_value >= 0.5 else 0
        elif method == "any":
            return 1 if agg_value > 0 else 0
        else:
            raise ValueError(f"Unsupported method: {method}")
    merged_df['polarized'] = merged_df['polarized'].fillna(0)
    merged_df['polarized_value'] = merged_df.apply(lambda row: process_polarized_values(row, method), axis=1)

    # Format text based on language and add polarization
    def format_text(row, language):
        if language == "id":
            return f"Nilai polarisasi rata-rata (rentang 0 hingga 1): {row['polarized_value']} [SEP] {row['text']}"
        elif language == "en":
            return f"Average polarization value (range of 0 to 1): {row['polarized_value']} [SEP] {row['text']}"
        else:
            raise ValueError(f"Unsupported language: {language}")

    merged_df['combined_text'] = merged_df.apply(lambda row: format_text(row, language), axis=1)

    # Prepare outputs
    texts = merged_df['combined_text'].tolist()
    labels = merged_df['label'].tolist()

    return texts, labels


def train_featural_bert_pipeline_with_polarized_feature(model_path: str, merged_df, output_dir: str,
                                                           method: str = "agg", language: str = "id", raw_test: bool = False):
    # Handle texts and labels
    texts, labels = featural_text_handler(merged_df, method, language)

    # Create output directory
    os.makedirs(output_dir, exist_ok=True)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    metrics_list = []

    for fold, (train_index, test_index) in enumerate(skf.split(texts, labels)):
        train_texts, test_texts = np.array(texts)[train_index], np.array(texts)[test_index]
        if raw_test:
            test_texts = [text.split('[SEP]')[-1].strip() for text in test_texts]
        train_labels, test_labels = np.array(labels)[train_index], np.array(labels)[test_index]

        # Tokenize
        tokenizer = BertTokenizer.from_pretrained(model_path)
        train_encodings = tokenizer(list(train_texts), truncation=True, padding=True, max_length=512, return_tensors='pt')
        test_encodings = tokenizer(list(test_texts), truncation=True, padding=True, max_length=512, return_tensors='pt')

        train_dataset = Dataset(train_encodings, train_labels)
        test_dataset = Dataset(test_encodings, test_labels)

        model = BertForSequenceClassification.from_pretrained(model_path, num_labels=len(set(labels)))

        training_args = TrainingArguments(
            output_dir=os.path.join(output_dir, f"model_fold_{fold}"),
            evaluation_strategy="epoch",
            per_device_train_batch_size=16,
            per_device_eval_batch_size=64,
            num_train_epochs=3,
            logging_dir=os.path.join(output_dir, f"logs_fold_{fold}"),
            save_strategy="no",
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=test_dataset,
            compute_metrics=compute_metrics,
        )

        # Train and evaluate
        trainer.train()
        metrics = trainer.evaluate()
        metrics_list.append(metrics)

        # Save performance report
        pd.DataFrame([metrics]).to_csv(os.path.join(output_dir, f"performance_fold_{fold}.csv"), index=False)

    # Calculate average performance metrics
    avg_metrics = {metric: np.mean([m[metric] for m in metrics_list]) for metric in metrics_list[0]}
    pd.DataFrame([avg_metrics]).to_csv(os.path.join(output_dir, "average_performance.csv"), index=False)

# Dataset class to handle encoding
class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = [int(label) for label in labels] 

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)


## 4. Incorporating Demographic Information
Some normalization are required before passing the values to the model.


Main function: `exploded_df_train_baseline_bert_pipeline_with_demographic_feature`

In [ ]:
import pandas as pd
import ast
import os
import numpy as np
from typing import List
from sklearn.model_selection import StratifiedKFold
from transformers import Trainer, TrainingArguments, BertForSequenceClassification, BertTokenizer
import torch

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, average_precision_score

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    # Accuracy
    accuracy = accuracy_score(labels, preds)

    # Macro F1, Precision, and Recall
    macro_f1 = f1_score(labels, preds, average='macro')
    precision = precision_score(labels, preds, average='macro')
    recall = recall_score(labels, preds, average='macro')

    # Class-1 only metrics (positive class)
    precision_class_1 = precision_score(labels, preds, pos_label=1)
    recall_class_1 = recall_score(labels, preds, pos_label=1)
    f1_class_1 = f1_score(labels, preds, pos_label=1)

    # Class-0 only metrics (negative class)
    precision_class_0 = precision_score(labels, preds, pos_label=0)
    recall_class_0 = recall_score(labels, preds, pos_label=0)
    f1_class_0 = f1_score(labels, preds, pos_label=0)

    # ROC-AUC score for binary classification
    try:
        # Compute the ROC AUC score for binary classification directly
        roc_auc = roc_auc_score(labels, preds)
    except ValueError:
        # In case there's an issue with the labels or predictions (e.g., all labels are the same)
        roc_auc = 0.5  # This would represent random classification if AUC can't be computed

    # Precision-Recall AUC
    precision_recall_auc = average_precision_score(labels, preds)

    return {
        'accuracy': accuracy,
        'macro_f1': macro_f1,
        'precision': precision,
        'recall': recall,
        'precision_class_1': precision_class_1,
        'recall_class_1': recall_class_1,
        'f1_class_1': f1_class_1,
        'precision_class_0': precision_class_0,
        'recall_class_0': recall_class_0,
        'f1_class_0': f1_class_0,
        'roc_auc': roc_auc,
        'precision_recall_auc': precision_recall_auc,
    }

def demographic_text_handler(merged_df):
    texts = merged_df['text'].tolist()
    labels = merged_df['label'].tolist()
    return merged_df, texts, labels

def single_level_demographic_text_handler(df, demographic: List[str] = [], language: str = "id"):
    """
    Handles polar and non-polar text datasets for polarity classification, 
    applying demographic information

    Language options:
        1. id = Indonesian.
        2. en = English.

    Demographic options, may be a list of strings:
        1. ethnicity
        2. religion
        3. disability
        4. lgbt
        5. gender
        6. age_group
        7. domisili
        8. pendidikan terakhir
        9. status pekerjaan
        10. president vote leaning
    """

    id_demographic_names = {
        'ethnicity': 'etnisitas',
        'religion' : 'agama',
        'disability': 'disabilitas',
        'lgbt' : 'lgbt',
        'gender': 'gender',
        'age_group': 'generasi',
        'domisili': 'domisili',
        'pendidikan terakhir': 'pendidikan terakhir',
        'status pekerjaan': 'status pekerjaan',
        'president vote leaning': 'pilihan presiden'
    }

    en_demographic_names = {
        'ethnicity': 'ethnicity',
        'religion': 'religion',
        'disability': 'disability',
        'lgbt': 'lgbt',
        'gender': 'gender',
        'age_group': 'generation',
        'domisili': 'domicile',
        'pendidikan terakhir': 'last education level',
        'status pekerjaan': 'job status',
        'president vote leaning': 'president vote leaning'
        
    }

    # preprocess toxic dataset
    df['label'] = df['toxicity']
    
    # Format text based on language and add polarization
    def format_text(row, language, demographic):
        if language == "id":
            if len(demographic) == 0:
                return "Informasi Demografis: Tidak tersedia"
            
            input_string = "Informasi Demografis:\n"
            for demo in demographic:
                input_string += f"{id_demographic_names[demo]}: {row[demo]}\n"
            input_string = input_string.strip("\n")
            input_string = f"{input_string} [SEP] {row['text']}"
            return input_string
        elif language == "en":
            if len(demographic) == 0:
                return "Demographic Information: Not available"

            input_string = "Demographic Information:\n"
            for demo in demographic:
                input_string += f"{en_demographic_names[demo]}: {row[demo]}\n"
            input_string = input_string.strip("\n")
            input_string = f"{input_string} [SEP] {row['text']}"
            return input_string
        else:
            raise ValueError(f"Unsupported language: {language}")

    df['combined_text'] = df.apply(lambda row: format_text(row, language, demographic), axis=1)
    print(df['label'].value_counts())

    # Prepare outputs
    texts = df['combined_text'].tolist()
    labels = df['label'].tolist()

    return texts, labels

import pandas as pd
import ast
def process_and_explode(df):
    def age_group_f(x):
        if 12 <= x <= 29:
            return "Gen Z"
        if 30 <= x <= 44:
            return "Millenials"
        if 45 <= x <= 59:
            return "Gen X"
            
    def president_vote_f(x):
        if x == "1":
            return "Anies Rasyid Baswedan-Muhaimin Iskandar"
        if x == "2":
            return "Prabowo Subianto-Gibran Rakabuming Raka"
        if x == "3":
            return "Ganjar Pranowo-Mahfud MD"
        return x    
        
    annotator_df = pd.read_json("hf://datasets/Exqrch/IndoToxic2024/indotoxic2024_annotator_demographic_data_v2_final.jsonl", lines=True)
    annotator_df['gender'] = annotator_df['gender'].apply(lambda x: x.strip())
    annotator_df['age_group'] = annotator_df['age'].astype(int)
    annotator_df['age_group'] = annotator_df['age_group'].apply(lambda x: age_group_f(x))
    annotator_df['status pekerjaan'] = annotator_df['status pekerjaan'].apply(lambda x: 'Tidak Bekerja' if x == "Ibu Rumah Tangga" else x)
    annotator_df['president vote leaning'] = annotator_df['president vote leaning'].apply(lambda x: "Tidak ada" if x not in ["1", "2", "3"] else x)
    annotator_df['president vote leaning'] = annotator_df['president vote leaning'].apply(lambda x : president_vote_f(x))   
    annotator_df['annotator_id'] = annotator_df['annotator_id'].astype(str)

    columns = [
        'is_noise_or_spam_text',
        'related_to_election_2024',
        'toxicity',
        'polarized',
        'profanity_obscenity',
        'threat_incitement_to_violence',
        'insults',
        'identity_attack',
        'sexually_explicit'
    ]

    df['annotators_id'] = df['annotators_id'].apply(lambda x: ast.literal_eval(x))
    df_exploded = df.explode('annotators_id')
    df_exploded.rename(columns={
        'annotators_id': 'annotator_id'
    }, inplace=True)
    merged_df = df_exploded.merge(annotator_df, on="annotator_id", how="inner")
    merged_df['text_id_index'] = merged_df.groupby('text_id').cumcount()
    merged_df['text_id_index'] = merged_df['text_id_index'].astype(int)
    for col in columns:
        merged_df[col] = merged_df[col].apply(lambda x: ast.literal_eval(x))
        merged_df[col] = merged_df.apply(lambda row: row[col][row['text_id_index']], axis=1)

    return merged_df
    
def exploded_df_train_baseline_bert_pipeline_with_demographic_feature(model_path: str, 
                                                                      merged_df, 
                                                                      output_dir: str,
                                                                      demographic: List[str] = [], 
                                                                      language: str = "id",
                                                                      raw_test: bool = False):
    # Handle texts and labels
    original_df, texts, labels = demographic_text_handler(merged_df) # Just using this to ensure replicability with old baseline

    # Create output directory
    os.makedirs(output_dir, exist_ok=True)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    metrics_list = []

    for fold, (train_index, test_index) in enumerate(skf.split(texts, labels)):
        train_df = original_df.iloc[train_index]
        test_df = original_df.iloc[test_index]

        train_df = process_and_explode(train_df)
        test_df = process_and_explode(test_df)

        train_texts, train_labels = single_level_demographic_text_handler(train_df, demographic, language)
        test_texts, test_labels = single_level_demographic_text_handler(test_df, demographic, language)
        
        if raw_test:
            test_texts = [text.split('[SEP]')[-1].strip() for text in test_texts]

        train_labels = [int(x) for x in train_labels]
        test_labels = [int(x) for x in test_labels]

        # Tokenize
        tokenizer = BertTokenizer.from_pretrained(model_path)
        train_encodings = tokenizer(list(train_texts), truncation=True, padding=True, max_length=512, return_tensors='pt')
        test_encodings = tokenizer(list(test_texts), truncation=True, padding=True, max_length=512, return_tensors='pt')

        train_dataset = Dataset(train_encodings, train_labels)
        test_dataset = Dataset(test_encodings, test_labels)

        model = BertForSequenceClassification.from_pretrained(model_path, num_labels=len(set(train_labels)))

        training_args = TrainingArguments(
            output_dir=os.path.join(output_dir, f"temp_model_fold_{fold}"),  # Temporary directory for Trainer
            evaluation_strategy="epoch",
            per_device_train_batch_size=16,
            per_device_eval_batch_size=64,
            num_train_epochs=3,
            logging_dir=os.path.join(output_dir, f"logs_fold_{fold}"),
            save_strategy="no",  # Prevent model saving during fold training
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=test_dataset,
            compute_metrics=compute_metrics,
        )

        # Train and evaluate
        trainer.train()
        metrics = trainer.evaluate()
        metrics_list.append(metrics)

        # Save performance report
        pd.DataFrame([metrics]).to_csv(os.path.join(output_dir, f"performance_fold_{fold}.csv"), index=False)

    # Calculate average performance metrics
    avg_metrics = {metric: np.mean([m[metric] for m in metrics_list]) for metric in metrics_list[0]}
    pd.DataFrame([avg_metrics]).to_csv(os.path.join(output_dir, "average_performance.csv"), index=False)

# Dataset class to handle encoding
class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = [int(label) for label in labels] 

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)


## 5. Combining Polarization and Demographic Information for Toxicity Detection
Main Function: `exploded_df_train_baseline_bert_pipeline_with_polarization_and_demographic_feature`

In [ ]:
import pandas as pd
import ast
import os
import numpy as np
from typing import List
from sklearn.model_selection import StratifiedKFold
from transformers import Trainer, TrainingArguments, BertForSequenceClassification, BertTokenizer
import torch

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, average_precision_score

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    # Accuracy
    accuracy = accuracy_score(labels, preds)

    # Macro F1, Precision, and Recall
    macro_f1 = f1_score(labels, preds, average='macro')
    precision = precision_score(labels, preds, average='macro')
    recall = recall_score(labels, preds, average='macro')

    # Class-1 only metrics (positive class)
    precision_class_1 = precision_score(labels, preds, pos_label=1)
    recall_class_1 = recall_score(labels, preds, pos_label=1)
    f1_class_1 = f1_score(labels, preds, pos_label=1)

    # Class-0 only metrics (negative class)
    precision_class_0 = precision_score(labels, preds, pos_label=0)
    recall_class_0 = recall_score(labels, preds, pos_label=0)
    f1_class_0 = f1_score(labels, preds, pos_label=0)

    # ROC-AUC score for binary classification
    try:
        # Compute the ROC AUC score for binary classification directly
        roc_auc = roc_auc_score(labels, preds)
    except ValueError:
        # In case there's an issue with the labels or predictions (e.g., all labels are the same)
        roc_auc = 0.5  # This would represent random classification if AUC can't be computed

    # Precision-Recall AUC
    precision_recall_auc = average_precision_score(labels, preds)

    return {
        'accuracy': accuracy,
        'macro_f1': macro_f1,
        'precision': precision,
        'recall': recall,
        'precision_class_1': precision_class_1,
        'recall_class_1': recall_class_1,
        'f1_class_1': f1_class_1,
        'precision_class_0': precision_class_0,
        'recall_class_0': recall_class_0,
        'f1_class_0': f1_class_0,
        'roc_auc': roc_auc,
        'precision_recall_auc': precision_recall_auc,
    }

def single_level_toxicity_and_demographic_text_handler(df, demographic: List[str] = [], language: str = "id"):
    """
    Handles polar and non-polar text datasets for polarity classification, 
    applying demographic information

    Language options:
        1. id = Indonesian.
        2. en = English.

    Demographic options, may be a list of strings:
        1. ethnicity
        2. religion
        3. disability
        4. lgbt
        5. gender
        6. age_group
        7. domisili
        8. pendidikan terakhir
        9. status pekerjaan
        10. president vote leaning
    """

    id_demographic_names = {
        'ethnicity': 'etnisitas',
        'religion' : 'agama',
        'disability': 'disabilitas',
        'lgbt' : 'lgbt',
        'gender': 'gender',
        'age_group': 'generasi',
        'domisili': 'domisili',
        'pendidikan terakhir': 'pendidikan terakhir',
        'status pekerjaan': 'status pekerjaan',
        'president vote leaning': 'pilihan presiden'
    }

    en_demographic_names = {
        'ethnicity': 'ethnicity',
        'religion': 'religion',
        'disability': 'disability',
        'lgbt': 'lgbt',
        'gender': 'gender',
        'age_group': 'generation',
        'domisili': 'domicile',
        'pendidikan terakhir': 'last education level',
        'status pekerjaan': 'job status',
        'president vote leaning': 'president vote leaning'
        
    }

    # preprocess toxic dataset
    df['label'] = df['toxicity']
    
    # Format text based on language and add polarization
    def format_text(row, language, demographic):
        if language == "id":
            if len(demographic) == 0:
                return "Informasi Demografis dan Toksisitas: Tidak tersedia"
            
            input_string = f"{row['combined_text']}\nInformasi Demografis:\n"
            for demo in demographic:
                input_string += f"{id_demographic_names[demo]}: {row[demo]}\n"
            input_string = input_string.strip("\n")
            input_string = f"{input_string} [SEP] {row['text']}"
            return input_string
        elif language == "en":
            if len(demographic) == 0:
                return "Demographic Information and Toxicity: Not available"

            input_string = f"{row['combined_text']}\nDemographic Information:\n"
            for demo in demographic:
                input_string += f"{en_demographic_names[demo]}: {row[demo]}\n"
            input_string = input_string.strip("\n")
            input_string = f"{input_string} [SEP] {row['text']}"
            return input_string
        else:
            raise ValueError(f"Unsupported language: {language}")

    df['combined_text'] = df.apply(lambda row: format_text(row, language, demographic), axis=1)
    print(df['label'].value_counts())

    # Prepare outputs
    texts = df['combined_text'].tolist()
    labels = df['label'].tolist()

    return texts, labels

import pandas as pd
import ast
def process_and_explode(df):
    def age_group_f(x):
        if 12 <= x <= 29:
            return "Gen Z"
        if 30 <= x <= 44:
            return "Millenials"
        if 45 <= x <= 59:
            return "Gen X"
            
    def president_vote_f(x):
        if x == "1":
            return "Anies Rasyid Baswedan-Muhaimin Iskandar"
        if x == "2":
            return "Prabowo Subianto-Gibran Rakabuming Raka"
        if x == "3":
            return "Ganjar Pranowo-Mahfud MD"
        return x    
        
    annotator_df = pd.read_json("hf://datasets/Exqrch/IndoToxic2024/indotoxic2024_annotator_demographic_data_v2_final.jsonl", lines=True)
    annotator_df['gender'] = annotator_df['gender'].apply(lambda x: x.strip())
    annotator_df['age_group'] = annotator_df['age'].astype(int)
    annotator_df['age_group'] = annotator_df['age_group'].apply(lambda x: age_group_f(x))
    annotator_df['status pekerjaan'] = annotator_df['status pekerjaan'].apply(lambda x: 'Tidak Bekerja' if x == "Ibu Rumah Tangga" else x)
    annotator_df['president vote leaning'] = annotator_df['president vote leaning'].apply(lambda x: "Tidak ada" if x not in ["1", "2", "3"] else x)
    annotator_df['president vote leaning'] = annotator_df['president vote leaning'].apply(lambda x : president_vote_f(x))   
    annotator_df['annotator_id'] = annotator_df['annotator_id'].astype(str)

    columns = [
        'is_noise_or_spam_text',
        'related_to_election_2024',
        'toxicity',
        'polarized',
        'profanity_obscenity',
        'threat_incitement_to_violence',
        'insults',
        'identity_attack',
        'sexually_explicit'
    ]

    df['annotators_id'] = df['annotators_id'].apply(lambda x: ast.literal_eval(x))
    df_exploded = df.explode('annotators_id')
    df_exploded.rename(columns={
        'annotators_id': 'annotator_id'
    }, inplace=True)
    merged_df = df_exploded.merge(annotator_df, on="annotator_id", how="inner")
    merged_df['text_id_index'] = merged_df.groupby('text_id').cumcount()
    merged_df['text_id_index'] = merged_df['text_id_index'].astype(int)
    for col in columns:
        merged_df[col] = merged_df[col].apply(lambda x: ast.literal_eval(x))
        merged_df[col] = merged_df.apply(lambda row: row[col][row['text_id_index']], axis=1)

    return merged_df

def polarity_text_handler(merged_df, method: str = "agg", language: str = "id"):
    """
    Handles toxic and non-toxic text datasets for toxicity classification, 
    applying polarization processing and text formatting.

    Method options:
        1. agg = Aggregate value with a range of [0, 1].
        2. bin = Binarized, values of either 0 or 1 (values of 0.5 converted to 0).
        3. bin-ceil = Binarized, but values of 0.5 converted to 1.
        4. any = Binarized, any value above 0 is converted to 1.

    Language options:
        1. id = Indonesian.
        2. en = English.
    """
    def process_polarized_values(row, method):
        """Processes the polarization values according to the selected method."""
        values = ast.literal_eval(row['polarized']) if isinstance(row['polarized'], str) else row['polarized']
        values = [int(x) for x in values]
        if not values:
            return 0  # Default for missing or empty polarization
        
        agg_value = sum(values) / len(values)
        if method == "agg":
            return agg_value
        elif method == "bin":
            return 1 if agg_value > 0.5 else 0
        elif method == "bin-ceil":
            return 1 if agg_value >= 0.5 else 0
        elif method == "any":
            return 1 if agg_value > 0 else 0
        else:
            raise ValueError(f"Unsupported method: {method}")

    merged_df['polarized'] = merged_df['polarized'].fillna(0)
    merged_df['polarized_value'] = merged_df.apply(lambda row: process_polarized_values(row, method), axis=1)

    # Format text based on language and add polarization
    def format_text(row, language):
        if language == "id":
            return f"Nilai polarisasi rata-rata (rentang 0 hingga 1): {row['polarized_value']}"
        elif language == "en":
            return f"Average polarization value (range of 0 to 1): {row['polarized_value']}"
        else:
            raise ValueError(f"Unsupported language: {language}")

    merged_df['combined_text'] = merged_df.apply(lambda row: format_text(row, language), axis=1)

    # Prepare outputs
    texts = merged_df['combined_text'].tolist()
    labels = merged_df['label'].tolist()

    return merged_df, texts, labels


def exploded_df_train_baseline_bert_pipeline_with_polarization_and_demographic_feature(model_path: str, 
                                                                      merged_df, 
                                                                      output_dir: str,
                                                                      demographic: List[str] = [], 
                                                                      method: str = "agg",
                                                                      language: str = "id",
                                                                      raw_test: bool = False):
    # Handle texts and labels
    original_df, texts, labels = polarity_text_handler(merged_df, method, language) # Just using this to ensure replicability with old baseline

    # Create output directory
    os.makedirs(output_dir, exist_ok=True)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    metrics_list = []

    for fold, (train_index, test_index) in enumerate(skf.split(texts, labels)):
        train_df = original_df.iloc[train_index]
        test_df = original_df.iloc[test_index]

        train_df = process_and_explode(train_df)
        test_df = process_and_explode(test_df)

        train_texts, train_labels = single_level_toxicity_and_demographic_text_handler(train_df, demographic, language)
        test_texts, test_labels = single_level_toxicity_and_demographic_text_handler(test_df, demographic, language)
        
        if raw_test:
            test_texts = [text.split('[SEP]')[-1].strip() for text in test_texts]

        train_labels = [int(x) for x in train_labels]
        test_labels  = [int(x) for x in test_labels]

        # Tokenize
        tokenizer = BertTokenizer.from_pretrained(model_path)
        train_encodings = tokenizer(list(train_texts), truncation=True, padding=True, max_length=512, return_tensors='pt')
        test_encodings = tokenizer(list(test_texts), truncation=True, padding=True, max_length=512, return_tensors='pt')

        train_dataset = Dataset(train_encodings, train_labels)
        test_dataset = Dataset(test_encodings, test_labels)

        model = BertForSequenceClassification.from_pretrained(model_path, num_labels=len(set(train_labels)))

        training_args = TrainingArguments(
            output_dir=os.path.join(output_dir, f"temp_model_fold_{fold}"),  # Temporary directory for Trainer
            evaluation_strategy="epoch",
            per_device_train_batch_size=16,
            per_device_eval_batch_size=64,
            num_train_epochs=3,
            logging_dir=os.path.join(output_dir, f"logs_fold_{fold}"),
            save_strategy="no",  # Prevent model saving during fold training
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=test_dataset,
            compute_metrics=compute_metrics,
        )

        # Train and evaluate
        trainer.train()
        metrics = trainer.evaluate()
        metrics_list.append(metrics)

        # Save performance report
        pd.DataFrame([metrics]).to_csv(os.path.join(output_dir, f"performance_fold_{fold}.csv"), index=False)

    # Calculate average performance metrics
    avg_metrics = {metric: np.mean([m[metric] for m in metrics_list]) for metric in metrics_list[0]}
    pd.DataFrame([avg_metrics]).to_csv(os.path.join(output_dir, "average_performance.csv"), index=False)

# Dataset class to handle encoding
class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = [int(label) for label in labels] 

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)